<div style="background:#0B3C49;color:white;padding:14px 18px;border-radius:8px">
<b style="color:#F2A541">INTERNATIONAL BALKAN UNIVERSITY · DS 501 INTRODUCTION TO DATA SCIENCE</b><br>
<span style="font-size:1.4em">Lab 5 · Data Cleaning, Wrangling and Exploratory Analysis</span><br>
Week 5 · Lessons 5-1 to 5-3 · about 90 minutes
</div>

## Learning goals
1. Detect quality problems: inconsistent codes, wrong types, invalid values, duplicates, missing data.
2. Choose and justify a strategy for missing values (MCAR, MAR, MNAR).
3. Wrangle data with pandas: recode, merge, group, and engineer features.
4. Explore relationships and recognise a confounder.

**Dataset:** Titanic passengers (891 rows, the standard teaching sample). We first create a deliberately **messy copy** that imitates problems found in real data.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
clean_src = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
clean_src.insert(0, "passenger_id", range(1, len(clean_src) + 1))

# --- create a messy copy (do not edit this cell) ---
rng = np.random.default_rng(501)
messy = clean_src.copy()
idx = rng.choice(len(messy), 60, replace=False)
messy.loc[idx[:20], "sex"] = rng.choice(["M", "F", " male", "Female ", "MALE"], 20)
messy["fare"] = messy["fare"].round(2).astype(str)
messy.loc[idx[20:35], "fare"] = messy.loc[idx[20:35], "fare"].str.replace(".", ",", regex=False) + " GBP"
messy.loc[idx[35:42], "age"] = rng.choice([-1, 0, 250, 999], 7)
messy.loc[idx[42:], "embark_town"] = messy.loc[idx[42:], "embark_town"].str.upper()
messy = pd.concat([messy, messy.sample(12, random_state=1)], ignore_index=True)   # true duplicates
print(messy.shape)
messy.sample(5, random_state=3)

## 1. Inspect before you clean

In [ ]:
messy.info()
print(messy["sex"].value_counts(dropna=False))
print(messy["embark_town"].value_counts(dropna=False))
print(messy["age"].describe())

**✏️ Your turn (written).** List every quality problem you can see, and name the quality dimension each one violates (completeness, validity, consistency, uniqueness).

*Your answer:* 

## 2. Fix consistency and types

**✏️ Your turn.** Standardise `sex` to exactly `male` / `female` and `embark_town` to title case. Show the value counts afterwards.

In [ ]:
# Hint: str.strip(), str.lower(), then map 'm'/'f'
df = messy.copy()
# YOUR CODE HERE

**✏️ Your turn.** Convert `fare` to a float: remove ' GBP', replace the decimal comma, and convert. Check that no value became NaN.

In [ ]:
# Hint: chain .str.replace(...) calls, then .astype(float)
# YOUR CODE HERE

**✏️ Your turn.** Remove **true duplicates** (same `passenger_id`). How many rows were removed?

In [ ]:
# Hint: drop_duplicates(subset='passenger_id')
# YOUR CODE HERE

In [ ]:
# Not every duplicate row is an error: identical attributes, different people
print("Rows identical on every column except passenger_id:", clean_src.drop(columns="passenger_id").duplicated().sum())

**✏️ Your turn (written).** Why would `df.drop_duplicates()` on all columns *except* the ID be a mistake here?

*Your answer:* 

**✏️ Your turn.** Set impossible ages (below 0 or above 100) to missing. Keep ages between 0 and 1 (infants recorded in months).

In [ ]:
# Hint: build a boolean mask and assign np.nan with .loc
# YOUR CODE HERE

## 3. Missing data

In [ ]:
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(5))
print(df.groupby("pclass")["age"].apply(lambda s: s.isna().mean()).round(2))

**✏️ Your turn (written).** Age is missing far more often in third class. Is age missing completely at random (MCAR), at random given other variables (MAR), or not at random (MNAR)? What does this imply for simply dropping those rows?

*Your answer:* 

In [ ]:
df["age_missing"] = df["age"].isna().astype(int)          # keep a flag: missingness can be informative
df["age"] = df.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))
df = df.drop(columns=["alive", "class", "deck"])           # redundant or mostly empty
print(df.isna().sum()[df.isna().sum() > 0])

## 4. Wrangling: features and merges

**✏️ Your turn.** Create `family_size` = sibsp + parch + 1, `is_child` (age < 16), and `fare_per_person` = fare / family_size.

In [ ]:
# YOUR CODE HERE

In [ ]:
ports = pd.DataFrame({"embarked": ["S", "C", "Q"], "port_country": ["England", "France", "Ireland"],
                      "boarding_order": [1, 2, 3]})
df = df.merge(ports, on="embarked", how="left", validate="many_to_one")
df[["embarked", "embark_town", "port_country"]].drop_duplicates()

## 5. Exploratory analysis

In [ ]:
print(df.groupby("sex")["survived"].mean().round(3))
pd.crosstab(df["pclass"], df["sex"], values=df["survived"], aggfunc="mean").round(3)

**✏️ Your turn.** Plot the age distribution of survivors and non-survivors on the same histogram, and compute the survival rate of children vs adults.

In [ ]:
# YOUR CODE HERE

In [ ]:
# A confounder: port of embarkation
print(df.groupby("embark_town")["survived"].mean().round(3))
pd.crosstab(df["embark_town"], df["pclass"], normalize="index").round(2)

**✏️ Your turn (written).** Passengers who boarded at Cherbourg survived more often. Did boarding in Cherbourg make people safer? Use the second table in your answer.

*Your answer:* 

**✏️ Your turn.** Check the confounding: compute survival by `embark_town` **within each** `pclass`.

In [ ]:
# YOUR CODE HERE

**✏️ Your turn (written).** Write three findings from this exploratory analysis that you would report to a historian, each with a number.

*Your answer:* 

---
### Submission checklist
- [ ] All ✏️ cells completed
- [ ] `Kernel → Restart & Run All` runs without errors
- [ ] Notebook saved and uploaded to Moodle before the next class